# Implicit Fixed-Step Batched ODE Solver (JAX + Colab GPU)

This notebook provides a batched implicit fixed-step BDF1 (Backward Euler) solver with Newton iterations.
It is designed for Google Colab GPU execution.


In [ ]:
# Colab GPU setup (run once, then Runtime -> Restart runtime)
# If JAX is already GPU-enabled, you can skip this.
%pip -q install -U "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


In [ ]:
import time
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# float64 required: newton_tol=1e-10 is unachievable in float32 (~1e-7 eps).
jax.config.update("jax_enable_x64", True)

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())
print('Default backend:', jax.default_backend())

In [ ]:
# ---------------------------------------------------------------------------
# Implicit fixed-step BDF1 (Backward Euler + Newton) — JAX/GPU batched solver
# ---------------------------------------------------------------------------

def rhs_implicit_fixed_single(t, y, lam):
    """Stiff test: y' = -lam*(y - cos(t)) - sin(t).
    Exact solution: y(t) = (y0 - 1)*exp(-lam*t) + cos(t).
    """
    return -lam * (y - jnp.cos(t)) - jnp.sin(t)


def jac_implicit_fixed_single(t, y, lam):
    """Analytic diagonal Jacobian: df/dy = -lam."""
    return -jnp.diag(lam)


def newton_be_batch(y_prev, t_next, h, lam, tol=1e-10, max_iter=20):
    """Batched Newton solve for BE residual: y - y_prev - h*f(t_next, y) = 0.

    Convergence is declared when inf-norm of residual < tol OR inf-norm of
    update delta < tol.  All trajectories run max_iter iterations; converged
    ones freeze (delta = 0) to avoid polluting results.
    """
    batch, dim = y_prev.shape
    eye = jnp.eye(dim)

    y = y_prev.copy()
    converged = jnp.zeros((batch,), dtype=bool)
    iters     = jnp.zeros((batch,), dtype=jnp.int32)

    for k in range(max_iter):
        f     = jax.vmap(rhs_implicit_fixed_single)(t_next, y, lam)
        res   = y - y_prev - h[:, None] * f
        res_n = jnp.max(jnp.abs(res), axis=1)

        just_conv = (~converged) & (res_n < tol)
        iters     = jnp.where(just_conv, k + 1, iters)
        converged = converged | just_conv

        if bool(jnp.all(converged)):
            break

        Jf    = jax.vmap(jac_implicit_fixed_single)(t_next, y, lam)
        A     = eye[None] - h[:, None, None] * Jf          # (B, d, d)
        delta = jnp.linalg.solve(A, -res[..., None]).squeeze(-1)
        # Freeze converged trajectories.
        delta = jnp.where(converged[:, None], 0.0, delta)
        y     = y + delta

        # Also declare convergence on small update.
        delta_n   = jnp.max(jnp.abs(delta), axis=1)
        just_conv2 = (~converged) & (delta_n < tol)
        iters      = jnp.where(just_conv2, k + 2, iters)
        converged  = converged | just_conv2

    iters = jnp.where((iters == 0) & converged, 1, iters)
    iters = jnp.where(~converged, max_iter, iters)
    return y, converged, iters


def solve_implicit_fixed_batch_jax(
    y0_batch,
    lam_batch,
    t_span=(0.0, 6.0),
    dt=0.02,
    newton_tol=1e-10,
    newton_max_iter=20,
    track_index=0,
):
    """Fixed-step Backward Euler solver over a batch of IVPs.

    The loop length is statically known (`n_steps`), so this is the ideal
    candidate for jax.lax.scan to eliminate Python-level overhead.
    Here we keep an explicit loop for clarity and Colab debuggability.
    """
    t0, tf = float(t_span[0]), float(t_span[1])
    y   = jnp.array(y0_batch, dtype=jnp.float64)
    lam = jnp.array(lam_batch, dtype=jnp.float64)

    batch   = y.shape[0]
    n_steps = int(np.ceil((tf - t0) / dt))

    t       = jnp.full((batch,), t0)
    success = jnp.ones((batch,), dtype=bool)
    n_newt  = jnp.zeros((batch,), dtype=jnp.int32)

    t_hist = [float(t[track_index])]
    y_hist = [np.array(y[track_index])]

    for _ in range(n_steps):
        h      = jnp.minimum(jnp.full((batch,), dt), tf - t)
        t_next = t + h

        y_next, conv, nit = newton_be_batch(
            y_prev=y, t_next=t_next, h=h, lam=lam,
            tol=newton_tol, max_iter=newton_max_iter,
        )
        y       = jnp.where(conv[:, None], y_next, y)
        t       = jnp.where(conv, t_next, t)
        success = success & conv
        n_newt  = n_newt + nit

        t_hist.append(float(t[track_index]))
        y_hist.append(np.array(y[track_index]))

    return {
        't_final':       np.array(t),
        'y_final':       np.array(y),
        'success':       np.array(success),
        'n_newton_total': np.array(n_newt),
        't_hist':        np.array(t_hist),
        'y_hist':        np.array(y_hist),
    }


def exact_implicit_fixed(t, y0, lam):
    """Exact solution: y(t) = (y0 - 1)*exp(-lam*t) + cos(t)."""
    return (y0 - 1.0) * np.exp(-lam * t) + np.cos(t)

In [ ]:
rng = np.random.default_rng(1)
batch_size = 512
dim = 4
t_span = (0.0, 6.0)
tf = t_span[1]
dt = 0.02

y0_batch  = rng.normal(size=(batch_size, dim))
lam_batch = rng.uniform(5.0, 60.0, size=(batch_size, dim))

cfg = dict(t_span=t_span, dt=dt, newton_tol=1e-10, newton_max_iter=20, track_index=0)

# --- Warmup ---
print("Warming up JAX...")
_ = solve_implicit_fixed_batch_jax(y0_batch[:4], lam_batch[:4], **cfg)

# --- Timed run ---
start = time.perf_counter()
out = solve_implicit_fixed_batch_jax(y0_batch, lam_batch, **cfg)
elapsed = time.perf_counter() - start

print(f"\nImplicit fixed-step BDF1 (JAX/GPU)  batch={batch_size}  dim={dim}  dt={dt}")
print(f"  Runtime           : {elapsed:.3f} s")
print(f"  Throughput        : {batch_size/elapsed:.0f} trajectories/s")
print(f"  Success rate      : {out['success'].mean()*100:.1f}%")
print(f"  Mean Newton total : {out['n_newton_total'].mean():.1f}")
n_steps = int(np.ceil((tf - t_span[0]) / dt))
print(f"  Mean Newton/step  : {out['n_newton_total'].mean() / n_steps:.2f}")

# --- Correctness vs exact solution ---
errors = []
for i in range(batch_size):
    if out['success'][i]:
        y_ex = exact_implicit_fixed(tf, y0_batch[i], lam_batch[i])
        errors.append(np.max(np.abs(out['y_final'][i] - y_ex)))
print(f"\n  Max |error| vs exact : {np.max(errors):.2e}")
print(f"  Mean|error| vs exact : {np.mean(errors):.2e}")
print(f"  (BDF1 is O(h); dt={dt} → expect ~O(1e-2) global error)")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(out['t_hist'], out['y_hist'][:, 0])
axes[0].set_xlabel('t'); axes[0].set_ylabel('y[0]')
axes[0].set_title('Tracked trajectory (implicit fixed-step BE, JAX)')
axes[0].grid(True)

axes[1].semilogy(sorted(errors))
axes[1].set_xlabel('Trajectory index (sorted)')
axes[1].set_ylabel('|error| at t=tf')
axes[1].set_title(f'Per-trajectory error vs exact  (dt={dt})')
axes[1].grid(True)
plt.tight_layout(); plt.show()